In [1]:
!pip install -q kaggle transformers datasets torch torchvision timm accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/cs231n_project"  # adjust to your folder
ZIP_PATH    = f"{PROJECT_DIR}/archive.zip"
DATA_DIR    = f"{PROJECT_DIR}/geo50k"

# unzip only if not done before
import pathlib, os, zipfile, shutil

if not pathlib.Path(DATA_DIR).exists():
    print("Unzipping dataset … this takes a few minutes.")
    pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATA_DIR)

!ls "$DATA_DIR" | head -5

Unzipping dataset … this takes a few minutes.
compressed_dataset


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/cs231n_project"  # adjust to your folder
ZIP_PATH    = f"{PROJECT_DIR}/archive.zip"
DATA_DIR    = f"{PROJECT_DIR}/geo50k"

In [3]:
%%writefile train_geo_drop_rare.py
"""
Fine-tune ResNet-50 on GeoGuessr images
(one sub-folder per country).

• Drop countries with fewer than --min-images pictures
  (default 5, change as needed).
• 80 / 10 / 10  splits:
      – train vs hold-out **stratified**
      – hold-out → val / test **random**.
• Supports --freeze and --log-every.
"""

import argparse, os, time, torch, evaluate
from collections import Counter
from datasets import load_dataset, Value
from transformers import (
    AutoImageProcessor, ResNetForImageClassification, default_data_collator,
    TrainingArguments, Trainer, logging as hf_logging
)

def ts():  # timestamp
    return time.strftime("[%H:%M:%S]")

# ───────────────────────── data loader ──────────────────────────
def load_filter_split(img_root: str, min_imgs: int, seed: int = 42):
    print(f"{ts()} Loading images from {img_root} …")
    ds = load_dataset("imagefolder", data_dir=img_root, split="train")
    print(f"{ts()} Raw dataset: {len(ds):,} images, "
          f"{ds.features['label'].num_classes} classes")

    # 1. drop rare countries
    cnt = Counter(ds["label"])
    keep_ids = {lbl for lbl, n in cnt.items() if n >= min_imgs}
    drop_ids = {lbl for lbl, n in cnt.items() if n <  min_imgs}
    if drop_ids:
        n_drop = sum(cnt[i] for i in drop_ids)
        print(f"{ts()} Dropping {len(drop_ids)} rare classes (<{min_imgs} imgs) "
              f"totalling {n_drop:,} images")
    ds = ds.filter(lambda ex: ex["label"] in keep_ids)

    # 2. rebuild ClassLabel for contiguous ids
    int2str = ds.features["label"].int2str
    ds = ds.map(lambda ex: {"label": int2str(ex["label"])})
    ds = ds.cast_column("label", Value("string"))
    ds = ds.class_encode_column("label")
    classes = ds.features["label"].names

    print(f"{ts()} Filtered dataset: {len(ds):,} images, "
          f"{len(classes)} classes after filtering")

    # 3. train / hold-out (stratified)
    try:
        split = ds.train_test_split(test_size=0.2, seed=seed,
                                    stratify_by_column="label")
    except ValueError as e:
        print(f"{ts()} ⚠️ Stratified split failed ({e}); using random split.")
        split = ds.train_test_split(test_size=0.2, seed=seed, shuffle=True)

    train = split["train"]
    hold  = split["test"]   # 20 %

    # 4. hold-out → val / test (random 50 / 50)
    split2 = hold.train_test_split(test_size=0.5, seed=seed, shuffle=True)
    val, test = split2["train"], split2["test"]

    return train, val, test, classes
# ─────────────────────────────────────────────────────────────────

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True,
                    help="Folder containing compressed_dataset/")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--freeze", action="store_true",
                    help="Train classifier head only")
    ap.add_argument("--log-every", type=int, default=25)
    ap.add_argument("--min-images", type=int, default=5)
    args = ap.parse_args()

    IMG_ROOT = os.path.join(args.root, "compressed_dataset")
    train, val, test, classes = load_filter_split(
        IMG_ROOT, min_imgs=args.min_images)

    proc  = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
    model = ResNetForImageClassification.from_pretrained(
        "microsoft/resnet-50",
        num_labels=len(classes),
        id2label=dict(enumerate(classes)),
        label2id={c: i for i, c in enumerate(classes)},
        ignore_mismatched_sizes=True,          # <-- NEW
    )

    if args.freeze:
        print(f"{ts()} Freezing backbone — only classifier will train")
        for p in model.parameters(): p.requires_grad = False
        for p in model.classifier.parameters(): p.requires_grad = True

    def tfm(ex):
        batch = proc(ex["image"], return_tensors="pt")
        batch["labels"] = ex["label"]
        return batch
    for ds in (train, val, test):
        ds.set_transform(tfm)

    hf_logging.set_verbosity_info()
    targs = TrainingArguments(
        output_dir="ckpt",
        num_train_epochs=args.epochs,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        #evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=args.log_every,
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
    )

    acc = evaluate.load("accuracy")
    def metrics(p):
        preds, refs = p.predictions, p.label_ids
        return {
            "accuracy": acc.compute(predictions=preds.argmax(1), references=refs)["accuracy"],
            "top5_accuracy": (preds.argsort(axis=-1)[:, -5:] == refs[:, None]).any(-1).mean(),
        }

    print(f"{ts()} Starting training for {args.epochs} epoch(s)…")
    trainer = Trainer(
        model,
        targs,
        train_dataset=train,
        eval_dataset=val,
        data_collator=default_data_collator,   # <-- new
        compute_metrics=metrics,
    )
    trainer.train()
    trainer.save_model("ckpt")

    print(f"{ts()} Training done — evaluating on test set")
    print(trainer.predict(test).metrics)

if __name__ == "__main__":
    main()


Writing train_geo_drop_rare.py


In [ ]:
!python train_geo_drop_rare.py \
        --root /content/drive/MyDrive/cs231n_project/geo50k \
        --epochs 3 \
        --log-every 10        # adjust as you like
        # --freeze            # uncomment for head-only baseline

2025-05-16 21:36:00.254471: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747431360.274835    2148 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747431360.280975    2148 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-16 21:36:00.303183: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[21:36:09] Loading images from /content/drive/MyDrive/cs231n_project/geo50k/compressed_dataset …
Resolving data files

In [5]:
from transformers import default_data_collator

KeyboardInterrupt: 

In [6]:
!pip install -q --upgrade "fsspec>=2024.3.0" datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
!pip install -q --upgrade "transformers>=4.40.0"

In [ ]:
import transformers, importlib, sys, pkg_resources, torch
print("Transformers version:", transformers.__version__)
print("Transformers path   :", importlib.util.find_spec("transformers").origin)

Transformers version: 4.51.3
Transformers path   : /usr/local/lib/python3.11/dist-packages/transformers/__init__.py
